# Nghiem thu RT-DETR 5-class (best5class.pt)

Notebook nay nghiem thu theo huong doanh nghiep: metric day du, toc do, do ben (robustness),
va ket luan PASS/WARN/FAIL theo muc tieu "kha tot".

Khong fake so lieu: bao cao dua tren ket qua thuc te cua model va dataset.


In [ ]:
!pip -q install -U "ultralytics>=8.4.20" "pyyaml>=6.0" "opencv-python-headless>=4.8.0" "tqdm>=4.66"

import json
import math
import os
import random
import shutil
import time
import unicodedata
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from ultralytics import RTDETR

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available(), "| gpus:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"gpu{i}:", torch.cuda.get_device_name(i))


In [ ]:
# ===== Config =====
MODEL_FILE_NAME = "best5class.pt"
MODEL_HINT = "best5class"                # if exact file name not found
DATASET_HINT = "rdd"                      # keyword for dataset auto-discovery

# Eval config
EVAL_IMGSZ = 1024
EVAL_BATCH = 2
EVAL_DEVICE = 0 if torch.cuda.is_available() else "cpu"
IOU_FOR_VAL = 0.7

# Deployment-oriented conf (for speed/robust tests only)
DEPLOY_CONF = 0.25

# Benchmark config
LATENCY_SAMPLE_IMAGES = 300
LATENCY_BATCH = 4
ROBUST_SAMPLE_IMAGES = 120

# Acceptance target (muc "kha tot")
TARGET = {
    "map50": 0.65,
    "map5095": 0.40,
    "precision": 0.70,
    "recall": 0.65,
    "latency_ms": 90.0,          # lower is better
    "robustness_drop": 0.25,     # lower is better (25% drop)
}

# Outputs
WORK_ROOT = Path("/kaggle/working")
RUNTIME_YAML = WORK_ROOT / "accept_runtime.yaml"
REPORT_JSON = WORK_ROOT / "acceptance_report.json"
REPORT_MD = WORK_ROOT / "acceptance_report.md"
PER_CLASS_CSV = WORK_ROOT / "acceptance_per_class_primary.csv"
CONFUSION_CSV = WORK_ROOT / "acceptance_confusion_primary.csv"
DATASET_SUMMARY_JSON = WORK_ROOT / "dataset_profile_summary.json"
DATASET_OVERVIEW_PNG = WORK_ROOT / "dataset_profile_overview.png"
CLASS_DIST_PNG = WORK_ROOT / "dataset_profile_class_dist.png"
BBOX_HIST_PNG = WORK_ROOT / "dataset_profile_bbox_hist.png"


In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def normalize_text(s: str) -> str:
    s = str(s).lower().strip()
    s = "".join(ch for ch in unicodedata.normalize("NFD", s) if unicodedata.category(ch) != "Mn")
    return s


def read_yaml(path: Path):
    return yaml.safe_load(path.read_text(encoding="utf-8"))


def normalize_names(names_raw):
    if isinstance(names_raw, list):
        return {i: str(v) for i, v in enumerate(names_raw)}
    if isinstance(names_raw, dict):
        out = {}
        for k, v in names_raw.items():
            out[int(k)] = str(v)
        return dict(sorted(out.items(), key=lambda x: x[0]))
    return None


def resolve_split_path(data_yaml_path: Path, cfg: dict, split_key: str):
    raw = cfg.get(split_key)
    if raw is None:
        return None
    p = Path(str(raw))
    if p.is_absolute():
        return p
    root = cfg.get("path")
    if root:
        rp = Path(str(root))
        if not rp.is_absolute():
            rp = (data_yaml_path.parent / rp).resolve()
        return (rp / p).resolve()
    return (data_yaml_path.parent / p).resolve()


def list_images(image_dir: Path):
    if image_dir is None or (not image_dir.exists()):
        return []
    return sorted([x for x in image_dir.rglob("*") if x.is_file() and x.suffix.lower() in IMG_EXTS])


def find_model_path(file_name="best5class.pt", hint=""):
    cands = []
    for base in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if not base.exists():
            continue
        for p in base.rglob("*.pt"):
            score = 0
            if p.name == file_name:
                score += 100
            s = normalize_text(p.as_posix())
            if hint and normalize_text(hint) in s:
                score += 20
            if "best" in p.name.lower():
                score += 5
            cands.append((score, len(str(p)), p.resolve()))
    if not cands:
        raise FileNotFoundError("No .pt model found under /kaggle/input or /kaggle/working")
    cands.sort(key=lambda x: (-x[0], x[1]))
    return cands[0][2]


def discover_dataset_from_yaml(dataset_hint="rdd"):
    cands = []
    for base in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if not base.exists():
            continue
        for p in base.rglob("data.yaml"):
            try:
                cfg = read_yaml(p)
            except Exception:
                continue
            if not isinstance(cfg, dict):
                continue
            val_key = "val" if "val" in cfg else ("valid" if "valid" in cfg else None)
            if "train" not in cfg or val_key is None:
                continue
            tr = resolve_split_path(p, cfg, "train")
            va = resolve_split_path(p, cfg, val_key)
            if tr is None or va is None or (not tr.exists()) or (not va.exists()):
                continue
            s = normalize_text(p.as_posix())
            score = 0
            if dataset_hint and normalize_text(dataset_hint) in s:
                score += 10
            if "rdd" in s:
                score += 3
            cands.append((score, len(s), p.resolve(), cfg, val_key))
    if not cands:
        return None
    cands.sort(key=lambda x: (-x[0], x[1]))
    _, _, yaml_path, cfg, val_key = cands[0]
    split_src = {
        "train": resolve_split_path(yaml_path, cfg, "train"),
        "val": resolve_split_path(yaml_path, cfg, val_key),
        "test": resolve_split_path(yaml_path, cfg, "test") if "test" in cfg else None,
    }
    return {
        "mode": "data_yaml",
        "yaml_path": yaml_path,
        "cfg": cfg,
        "split_src": split_src,
        "names": normalize_names(cfg.get("names")),
        "nc": int(cfg.get("nc", 0)) if cfg.get("nc") is not None else None,
    }


def classify_split_name(name: str):
    n = normalize_text(name)
    if any(k in n for k in ["train", "xe lua", "huan luyen"]):
        return "train"
    if any(k in n for k in ["val", "valid", "validation", "gia tri"]):
        return "val"
    if any(k in n for k in ["test", "kiem tra", "bai kiem tra"]):
        return "test"
    return None


def discover_dataset_from_dirs(dataset_hint="rdd"):
    split_dirs = []
    for base in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if not base.exists():
            continue
        for d in base.rglob("*"):
            if not d.is_dir():
                continue
            if (d / "images").exists() and (d / "labels").exists():
                split_dirs.append(d.resolve())

    if not split_dirs:
        return None

    mapped = {}
    for d in split_dirs:
        kind = classify_split_name(d.name)
        if kind is None:
            continue
        cur = mapped.get(kind)
        n_img = len(list_images(d / "images"))
        if cur is None or n_img > cur[0]:
            mapped[kind] = (n_img, d)

    if "train" not in mapped or "val" not in mapped:
        return None

    split_src = {
        "train": (mapped["train"][1] / "images"),
        "val": (mapped["val"][1] / "images"),
        "test": (mapped["test"][1] / "images") if "test" in mapped else None,
    }

    return {
        "mode": "split_dirs",
        "yaml_path": None,
        "cfg": {},
        "split_src": split_src,
        "names": None,
        "nc": None,
    }


def discover_dataset(dataset_hint="rdd"):
    info = discover_dataset_from_yaml(dataset_hint)
    if info is not None:
        return info
    info = discover_dataset_from_dirs(dataset_hint)
    if info is not None:
        return info
    raise FileNotFoundError("Cannot discover dataset splits (train/val[/test])")


model_path = find_model_path(MODEL_FILE_NAME, MODEL_HINT)
ds = discover_dataset(DATASET_HINT)

# Load model first, fallback names from model if dataset yaml does not provide names.
model = RTDETR(str(model_path))
model_names = {int(k): str(v) for k, v in model.names.items()} if hasattr(model, "names") else {0: "class0"}

names = ds["names"] if ds["names"] else model_names
nc = ds["nc"] if ds["nc"] else len(names)

# Runtime yaml for consistent val/test calls
runtime = {
    "path": str(Path(ds["split_src"]["train"]).parent.parent),
    "train": str(Path(ds["split_src"]["train"]).relative_to(Path(ds["split_src"]["train"]).parent.parent)),
    "val": str(Path(ds["split_src"]["val"]).relative_to(Path(ds["split_src"]["val"]).parent.parent)),
    "nc": int(nc),
    "names": {int(k): str(v) for k, v in sorted(names.items(), key=lambda x: int(x[0]))},
}
if ds["split_src"].get("test") is not None:
    runtime["test"] = str(Path(ds["split_src"]["test"]).relative_to(Path(ds["split_src"]["test"]).parent.parent))

# If splits are not under same root, use absolute path form directly
base_root = Path(ds["split_src"]["train"]).parent.parent
same_root = True
for k in ["val", "test"]:
    p = ds["split_src"].get(k)
    if p is not None and base_root not in p.parents and p != base_root:
        same_root = False
if not same_root:
    runtime = {
        "train": str(ds["split_src"]["train"]),
        "val": str(ds["split_src"]["val"]),
        "nc": int(nc),
        "names": {int(k): str(v) for k, v in sorted(names.items(), key=lambda x: int(x[0]))},
    }
    if ds["split_src"].get("test") is not None:
        runtime["test"] = str(ds["split_src"]["test"])

RUNTIME_YAML.write_text(yaml.safe_dump(runtime, sort_keys=False, allow_unicode=False), encoding="utf-8")

print("model_path:", model_path)
print("dataset_mode:", ds["mode"])
print("split_src:", {k: (str(v) if v else None) for k, v in ds["split_src"].items()})
print("runtime_yaml:", RUNTIME_YAML)
print("nc:", nc)
print("names:", names)


In [ ]:
# Dataset profile (for acceptance traceability)

def parse_label_for_stats(label_path: Path):
    boxes = []
    invalid = 0
    if not label_path.exists():
        return boxes, invalid

    for raw in label_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 5:
            invalid += 1
            continue
        try:
            cls = int(float(parts[0]))
            vals = [float(x) for x in parts[1:]]
        except Exception:
            invalid += 1
            continue

        if len(vals) == 4:
            box = normalize_bbox(vals)
        elif len(vals) >= 6 and len(vals) % 2 == 0:
            box = normalize_bbox(yolo_seg_to_bbox(vals))
        else:
            invalid += 1
            continue

        if box[2] <= 1e-9 or box[3] <= 1e-9:
            invalid += 1
            continue
        boxes.append((cls, box[0], box[1], box[2], box[3]))

    return boxes, invalid


def resolve_runtime_split(runtime_cfg: dict, split_key: str):
    raw = runtime_cfg.get(split_key)
    if raw is None:
        return None
    p = Path(str(raw))
    if p.is_absolute():
        return p
    root = runtime_cfg.get("path", None)
    if root:
        return (Path(str(root)) / p).resolve()
    return p.resolve()


split_paths = {
    "train": resolve_runtime_split(runtime, "train"),
    "val": resolve_runtime_split(runtime, "val"),
    "test": resolve_runtime_split(runtime, "test") if "test" in runtime else None,
}

dataset_summary = {}
all_area = []
all_aspect = []
all_class_counter = {int(i): 0 for i in sorted(runtime["names"].keys())}

for split_name, img_dir in split_paths.items():
    if img_dir is None or (not img_dir.exists()):
        continue

    lbl_dir = img_dir.parent / "labels"
    imgs = list_images(img_dir)
    pos_images = 0
    neg_images = 0
    invalid_lines = 0
    box_count = 0
    cls_counter = {int(i): 0 for i in sorted(runtime["names"].keys())}

    for img_path in tqdm(imgs, desc=f"profile {split_name}"):
        rel = img_path.relative_to(img_dir)
        lbl = lbl_dir / rel.with_suffix(".txt")
        boxes, bad = parse_label_for_stats(lbl)
        invalid_lines += int(bad)

        if boxes:
            pos_images += 1
            box_count += len(boxes)
            for cls, _, _, w, h in boxes:
                cls_counter[int(cls)] = cls_counter.get(int(cls), 0) + 1
                all_class_counter[int(cls)] = all_class_counter.get(int(cls), 0) + 1
                area = float(w * h)
                all_area.append(area)
                all_aspect.append(float(w / max(h, 1e-9)))
        else:
            neg_images += 1

    dataset_summary[split_name] = {
        "images": int(len(imgs)),
        "pos_images": int(pos_images),
        "neg_images": int(neg_images),
        "boxes": int(box_count),
        "invalid_label_lines": int(invalid_lines),
        "class_boxes": cls_counter,
    }

# Save summary json
DATASET_SUMMARY_JSON.write_text(json.dumps(dataset_summary, indent=2), encoding="utf-8")

# Plot 1: split overview
splits = [k for k in ["train", "val", "test"] if k in dataset_summary]
img_counts = [dataset_summary[s]["images"] for s in splits]
pos_counts = [dataset_summary[s]["pos_images"] for s in splits]
neg_counts = [dataset_summary[s]["neg_images"] for s in splits]

plt.figure(figsize=(9, 5))
x = np.arange(len(splits))
width = 0.25
plt.bar(x - width, img_counts, width=width, label="images")
plt.bar(x, pos_counts, width=width, label="pos_images")
plt.bar(x + width, neg_counts, width=width, label="neg_images")
plt.xticks(x, splits)
plt.ylabel("count")
plt.title("Dataset split overview")
plt.legend()
plt.tight_layout()
plt.savefig(DATASET_OVERVIEW_PNG, dpi=150)
plt.close()

# Plot 2: class box distribution
cls_ids = sorted(runtime["names"].keys())
cls_names = [str(runtime["names"][i]) for i in cls_ids]
cls_vals = [all_class_counter.get(int(i), 0) for i in cls_ids]

plt.figure(figsize=(10, 5))
plt.bar(cls_names, cls_vals)
plt.ylabel("box count")
plt.title("Class distribution (all profiled splits)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(CLASS_DIST_PNG, dpi=150)
plt.close()

# Plot 3: bbox area histogram (log scale)
plt.figure(figsize=(9, 5))
if len(all_area) > 0:
    area_arr = np.array(all_area, dtype=float)
    area_arr = np.clip(area_arr, 1e-9, None)
    plt.hist(np.log10(area_arr), bins=60)
    plt.xlabel("log10(box area normalized)")
    plt.ylabel("count")
    plt.title("BBox area histogram")
else:
    plt.text(0.5, 0.5, "No bbox found", ha="center", va="center")
    plt.title("BBox area histogram")
plt.tight_layout()
plt.savefig(BBOX_HIST_PNG, dpi=150)
plt.close()

dataset_plots = {
    "overview": str(DATASET_OVERVIEW_PNG),
    "class_distribution": str(CLASS_DIST_PNG),
    "bbox_area_hist": str(BBOX_HIST_PNG),
}

print("Dataset summary:")
print(json.dumps(dataset_summary, indent=2))
print("Dataset plots:", json.dumps(dataset_plots, indent=2))


In [ ]:
def safe_float(x, default=None):
    try:
        return float(x)
    except Exception:
        return default


def pack_metrics(metrics):
    p = safe_float(getattr(metrics.box, "mp", None), 0.0)
    r = safe_float(getattr(metrics.box, "mr", None), 0.0)
    m50 = safe_float(getattr(metrics.box, "map50", None), 0.0)
    m95 = safe_float(getattr(metrics.box, "map", None), 0.0)
    m75 = safe_float(getattr(metrics.box, "map75", None), None)
    f1 = (2.0 * p * r / (p + r + 1e-9)) if (p is not None and r is not None) else None
    return {
        "mAP50": m50,
        "mAP50-95": m95,
        "mAP75": m75,
        "precision": p,
        "recall": r,
        "f1": f1,
        "fitness": safe_float(getattr(metrics.box, "fitness", None), None),
    }


def extract_per_class(metrics, names_map):
    out = {}
    box = metrics.box
    maps = getattr(box, "maps", None)
    ap50 = getattr(box, "ap50", None)
    has_class_result = hasattr(box, "class_result")

    for i in sorted(names_map.keys()):
        name = str(names_map[i])
        item = {
            "precision": None,
            "recall": None,
            "f1": None,
            "mAP50": None,
            "mAP50-95": None,
        }
        if has_class_result:
            try:
                p_i, r_i, ap50_i, ap_i = box.class_result(i)
                item["precision"] = safe_float(p_i)
                item["recall"] = safe_float(r_i)
                if item["precision"] is not None and item["recall"] is not None:
                    item["f1"] = 2.0 * item["precision"] * item["recall"] / (item["precision"] + item["recall"] + 1e-9)
                item["mAP50"] = safe_float(ap50_i)
                item["mAP50-95"] = safe_float(ap_i)
            except Exception:
                pass

        if (item["mAP50"] is None) and (ap50 is not None) and (len(ap50) > i):
            item["mAP50"] = safe_float(ap50[i])
        if (item["mAP50-95"] is None) and (maps is not None) and (len(maps) > i):
            item["mAP50-95"] = safe_float(maps[i])

        out[name] = item
    return out


def extract_confusion(metrics, names_map):
    cm_obj = getattr(metrics, "confusion_matrix", None)
    if cm_obj is None:
        return None
    matrix = getattr(cm_obj, "matrix", None)
    if matrix is None:
        return None

    mat = np.array(matrix, dtype=float)
    if mat.ndim != 2:
        return None

    cls_names = [str(names_map[i]) for i in sorted(names_map.keys())]
    if mat.shape[0] == len(cls_names) + 1:
        labels = cls_names + ["background"]
    else:
        labels = [f"c{i}" for i in range(mat.shape[0])]

    n_cls = min(len(cls_names), mat.shape[0] - 1 if mat.shape[0] > len(cls_names) else mat.shape[0])
    per_cls = {}
    for i in range(n_cls):
        tp = mat[i, i]
        fp = mat[:, i].sum() - tp
        fn = mat[i, :].sum() - tp
        prec = tp / (tp + fp + 1e-9)
        rec = tp / (tp + fn + 1e-9)
        per_cls[labels[i]] = {
            "tp": float(tp),
            "fp": float(fp),
            "fn": float(fn),
            "precision": float(prec),
            "recall": float(rec),
        }

    return {
        "labels": labels,
        "matrix": mat.tolist(),
        "per_class": per_cls,
    }


def collect_artifacts(metrics):
    save_dir = Path(getattr(metrics, "save_dir", "")) if getattr(metrics, "save_dir", None) else None
    if save_dir is None or (not save_dir.exists()):
        return None
    files = [
        "confusion_matrix.png",
        "confusion_matrix_normalized.png",
        "PR_curve.png",
        "P_curve.png",
        "R_curve.png",
        "F1_curve.png",
        "results.png",
    ]
    out = {"save_dir": str(save_dir), "files": {}}
    for f in files:
        fp = save_dir / f
        out["files"][f] = str(fp) if fp.exists() else None
    return out


def run_eval(split_name):
    return model.val(
        data=str(RUNTIME_YAML),
        split=split_name,
        device=EVAL_DEVICE,
        imgsz=EVAL_IMGSZ,
        batch=EVAL_BATCH,
        conf=0.001,
        iou=IOU_FOR_VAL,
        plots=True,
        verbose=True,
    )


val_metrics = run_eval("val")
val_pack = pack_metrics(val_metrics)
val_per_class = extract_per_class(val_metrics, runtime["names"])
val_confusion = extract_confusion(val_metrics, runtime["names"])
val_artifacts = collect_artifacts(val_metrics)


test_pack = None
test_per_class = None
test_confusion = None
test_artifacts = None
if "test" in runtime:
    test_metrics = run_eval("test")
    test_pack = pack_metrics(test_metrics)
    test_per_class = extract_per_class(test_metrics, runtime["names"])
    test_confusion = extract_confusion(test_metrics, runtime["names"])
    test_artifacts = collect_artifacts(test_metrics)


print("VAL summary:", json.dumps(val_pack, indent=2))
if test_pack:
    print("TEST summary:", json.dumps(test_pack, indent=2))
print("VAL artifact dir:", val_artifacts["save_dir"] if val_artifacts else None)
print("TEST artifact dir:", test_artifacts["save_dir"] if test_artifacts else None)


In [ ]:
# Latency benchmark (deployment style)
val_img_dir = Path(runtime["val"])
if not val_img_dir.is_absolute():
    val_img_dir = Path(runtime.get("path", ".")) / val_img_dir

all_val_images = list_images(val_img_dir)
if len(all_val_images) == 0:
    raise SystemExit("No val images for latency benchmark")

rng = random.Random(SEED)
sample_n = min(LATENCY_SAMPLE_IMAGES, len(all_val_images))
bench_images = rng.sample(all_val_images, sample_n)

# warmup
_ = model.predict(source=[str(p) for p in bench_images[:min(8, len(bench_images))]], imgsz=EVAL_IMGSZ, conf=DEPLOY_CONF, device=EVAL_DEVICE, verbose=False)

def chunks(arr, n):
    for i in range(0, len(arr), n):
        yield arr[i:i+n]

latencies = []
for ch in tqdm(list(chunks(bench_images, LATENCY_BATCH)), desc="latency"):
    t0 = time.perf_counter()
    _ = model.predict(source=[str(p) for p in ch], imgsz=EVAL_IMGSZ, conf=DEPLOY_CONF, device=EVAL_DEVICE, verbose=False)
    dt = (time.perf_counter() - t0) * 1000.0 / max(1, len(ch))
    latencies.append(dt)

lat_arr = np.array(latencies, dtype=float)
latency_report = {
    "sample_images": int(sample_n),
    "batch": int(LATENCY_BATCH),
    "mean_ms_per_image": float(np.mean(lat_arr)),
    "p50_ms_per_image": float(np.percentile(lat_arr, 50)),
    "p95_ms_per_image": float(np.percentile(lat_arr, 95)),
    "fps_mean": float(1000.0 / max(1e-6, np.mean(lat_arr))),
}

print(json.dumps(latency_report, indent=2))


In [ ]:
# Robustness benchmark (brightness/blur/noise)
TMP_AUG = WORK_ROOT / "_accept_aug_tmp"
if TMP_AUG.exists():
    shutil.rmtree(TMP_AUG)
TMP_AUG.mkdir(parents=True, exist_ok=True)

val_img_dir = Path(runtime["val"])
if not val_img_dir.is_absolute():
    val_img_dir = Path(runtime.get("path", ".")) / val_img_dir
all_images = list_images(val_img_dir)
if len(all_images) == 0:
    raise SystemExit("No val images for robustness benchmark")

rng = random.Random(SEED + 1)
rob_n = min(ROBUST_SAMPLE_IMAGES, len(all_images))
rob_images = rng.sample(all_images, rob_n)


def infer_stats(img_path: Path):
    res = model.predict(source=str(img_path), imgsz=EVAL_IMGSZ, conf=DEPLOY_CONF, device=EVAL_DEVICE, verbose=False)[0]
    n = int(len(res.boxes)) if res.boxes is not None else 0
    if n == 0:
        return {"count": 0, "mean_conf": 0.0}
    conf = res.boxes.conf.detach().cpu().numpy()
    return {"count": int(n), "mean_conf": float(np.mean(conf))}


def aug_image(img, mode):
    if mode == "dark":
        return np.clip(img * 0.55, 0, 255).astype(np.uint8)
    if mode == "bright":
        return np.clip(img * 1.35, 0, 255).astype(np.uint8)
    if mode == "blur":
        return cv2.GaussianBlur(img, (5, 5), 0)
    if mode == "noise":
        noise = np.random.normal(0, 10, img.shape).astype(np.float32)
        return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    return img

mods = ["dark", "bright", "blur", "noise"]
rows = []
for img_path in tqdm(rob_images, desc="robust"):
    base = infer_stats(img_path)
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    for m in mods:
        aug = aug_image(img, m)
        out = TMP_AUG / f"{img_path.stem}_{m}.jpg"
        cv2.imwrite(str(out), aug)
        st = infer_stats(out)
        rows.append({
            "mode": m,
            "base_count": base["count"],
            "base_conf": base["mean_conf"],
            "aug_count": st["count"],
            "aug_conf": st["mean_conf"],
        })

robust = {}
for m in mods:
    sub = [r for r in rows if r["mode"] == m]
    if not sub:
        robust[m] = {"count_ratio": None, "conf_ratio": None}
        continue
    # count ratio
    base_c = np.array([r["base_count"] for r in sub], dtype=float)
    aug_c = np.array([r["aug_count"] for r in sub], dtype=float)
    count_ratio = float(np.mean((aug_c + 1e-6) / (base_c + 1e-6)))

    # conf ratio on cases where base has detections
    keep = [r for r in sub if r["base_count"] > 0]
    if keep:
        base_conf = np.array([r["base_conf"] for r in keep], dtype=float)
        aug_conf = np.array([r["aug_conf"] for r in keep], dtype=float)
        conf_ratio = float(np.mean((aug_conf + 1e-6) / (base_conf + 1e-6)))
    else:
        conf_ratio = 0.0

    robust[m] = {
        "count_ratio": count_ratio,
        "conf_ratio": conf_ratio,
        "count_drop": float(max(0.0, 1.0 - count_ratio)),
        "conf_drop": float(max(0.0, 1.0 - conf_ratio)),
    }

# mean drop across modes (using conf_drop when available)
valid_drops = [v["conf_drop"] for v in robust.values() if v.get("conf_drop") is not None]
robust_mean_drop = float(np.mean(valid_drops)) if valid_drops else 1.0

robust_report = {
    "sample_images": int(rob_n),
    "deploy_conf": float(DEPLOY_CONF),
    "by_mode": robust,
    "mean_conf_drop": robust_mean_drop,
}

print(json.dumps(robust_report, indent=2))


In [ ]:
# Acceptance scoring + report

def clamp01(x):
    return float(max(0.0, min(1.0, x)))


def score_higher_better(actual, target):
    if target <= 0:
        return 1.0
    return clamp01(actual / target)


def score_lower_better(actual, target):
    if actual <= 0:
        return 1.0
    return clamp01(target / actual)


# Use TEST metrics when available for final acceptance, else VAL
primary = test_pack if test_pack is not None else val_pack
primary_per_class = test_per_class if test_per_class is not None else val_per_class
primary_confusion = test_confusion if test_confusion is not None else val_confusion
primary_artifacts = test_artifacts if test_artifacts is not None else val_artifacts
primary_split = "test" if test_pack is not None else "val"

scores = {
    "map50": score_higher_better(primary["mAP50"], TARGET["map50"]),
    "map5095": score_higher_better(primary["mAP50-95"], TARGET["map5095"]),
    "precision": score_higher_better(primary["precision"], TARGET["precision"]),
    "recall": score_higher_better(primary["recall"], TARGET["recall"]),
    "latency": score_lower_better(latency_report["mean_ms_per_image"], TARGET["latency_ms"]),
    "robustness": score_lower_better(robust_report["mean_conf_drop"], TARGET["robustness_drop"]),
}

weights = {
    "map50": 0.25,
    "map5095": 0.25,
    "precision": 0.15,
    "recall": 0.15,
    "latency": 0.10,
    "robustness": 0.10,
}

overall = float(sum(scores[k] * weights[k] for k in scores.keys()))
if overall >= 0.85:
    level = "PASS_KHA_TOT"
elif overall >= 0.70:
    level = "WARN_CAN_CAI_THIEN"
else:
    level = "FAIL_DUOI_MUC_KHA"

# Gap-to-target (simulation, no fake metric)
gap = {
    "map50_need_delta": float(max(0.0, TARGET["map50"] - primary["mAP50"])),
    "map5095_need_delta": float(max(0.0, TARGET["map5095"] - primary["mAP50-95"])),
    "precision_need_delta": float(max(0.0, TARGET["precision"] - primary["precision"])),
    "recall_need_delta": float(max(0.0, TARGET["recall"] - primary["recall"])),
    "latency_need_improve_ms": float(max(0.0, latency_report["mean_ms_per_image"] - TARGET["latency_ms"])),
    "robustness_need_improve_drop": float(max(0.0, robust_report["mean_conf_drop"] - TARGET["robustness_drop"])),
}

report = {
    "model_path": str(model_path),
    "runtime_yaml": str(RUNTIME_YAML),
    "dataset_mode": ds["mode"],
    "split_src": {k: (str(v) if v else None) for k, v in ds["split_src"].items()},
    "dataset_profile": {
        "summary": dataset_summary,
        "plots": dataset_plots,
        "summary_json": str(DATASET_SUMMARY_JSON),
    },
    "kpi_target_kha_tot": TARGET,
    "val": {
        "summary": val_pack,
        "per_class": val_per_class,
        "confusion": val_confusion,
        "artifacts": val_artifacts,
    },
    "test": {
        "summary": test_pack,
        "per_class": test_per_class,
        "confusion": test_confusion,
        "artifacts": test_artifacts,
    } if test_pack else None,
    "latency": latency_report,
    "robustness": robust_report,
    "scores": scores,
    "overall_score": overall,
    "acceptance_level": level,
    "primary_split": primary_split,
    "gap_to_kha_tot": gap,
}

REPORT_JSON.write_text(json.dumps(report, indent=2), encoding="utf-8")

# Export per-class table (primary split)
per_rows = []
for cls_name, item in primary_per_class.items():
    per_rows.append({
        "split": primary_split,
        "class": cls_name,
        "precision": item.get("precision"),
        "recall": item.get("recall"),
        "f1": item.get("f1"),
        "mAP50": item.get("mAP50"),
        "mAP50-95": item.get("mAP50-95"),
    })

with PER_CLASS_CSV.open("w", newline="", encoding="utf-8") as f:
    import csv
    w = csv.DictWriter(f, fieldnames=["split", "class", "precision", "recall", "f1", "mAP50", "mAP50-95"])
    w.writeheader()
    for r in per_rows:
        w.writerow(r)

# Export confusion matrix (primary split)
if primary_confusion is not None:
    labels = primary_confusion["labels"]
    mat = np.array(primary_confusion["matrix"], dtype=float)
    with CONFUSION_CSV.open("w", newline="", encoding="utf-8") as f:
        import csv
        w = csv.writer(f)
        w.writerow(["true\pred"] + labels)
        for i, row in enumerate(mat.tolist()):
            w.writerow([labels[i]] + row)

md_lines = []
md_lines.append("# Acceptance Report - RT-DETR 5-class")
md_lines.append("")
md_lines.append(f"- Model: `{model_path}`")
md_lines.append(f"- Runtime YAML: `{RUNTIME_YAML}`")
md_lines.append(f"- Primary split: **{primary_split}**")
md_lines.append(f"- Acceptance level: **{level}**")
md_lines.append(f"- Overall score: **{overall:.3f}**")
md_lines.append("")
md_lines.append("## KPI (Primary split)")
md_lines.append(f"- mAP50: {primary['mAP50']:.4f} (target {TARGET['map50']})")
md_lines.append(f"- mAP50-95: {primary['mAP50-95']:.4f} (target {TARGET['map5095']})")
md_lines.append(f"- mAP75: {primary.get('mAP75')}")
md_lines.append(f"- Precision: {primary['precision']:.4f} (target {TARGET['precision']})")
md_lines.append(f"- Recall: {primary['recall']:.4f} (target {TARGET['recall']})")
md_lines.append(f"- F1: {primary.get('f1')}")
md_lines.append(f"- Latency mean: {latency_report['mean_ms_per_image']:.2f} ms/img (target <= {TARGET['latency_ms']})")
md_lines.append(f"- Robust mean conf drop: {robust_report['mean_conf_drop']:.4f} (target <= {TARGET['robustness_drop']})")
md_lines.append("")
md_lines.append("## Dataset profile")
md_lines.append(f"- Summary JSON: `{DATASET_SUMMARY_JSON}`")
md_lines.append(f"- Split overview plot: `{DATASET_OVERVIEW_PNG}`")
md_lines.append(f"- Class distribution plot: `{CLASS_DIST_PNG}`")
md_lines.append(f"- BBox area histogram: `{BBOX_HIST_PNG}`")
md_lines.append("")
md_lines.append("## Export files")
md_lines.append(f"- JSON: `{REPORT_JSON}`")
md_lines.append(f"- Markdown: `{REPORT_MD}`")
md_lines.append(f"- Per-class CSV: `{PER_CLASS_CSV}`")
md_lines.append(f"- Confusion CSV: `{CONFUSION_CSV}`")
if primary_artifacts is not None:
    md_lines.append(f"- Plot directory: `{primary_artifacts.get('save_dir')}`")
    for k, v in primary_artifacts.get("files", {}).items():
        md_lines.append(f"  - {k}: `{v}`")
md_lines.append("")
md_lines.append("## Gap to Kha Tot")
for k, v in gap.items():
    md_lines.append(f"- {k}: {v:.4f}")

REPORT_MD.write_text("\n".join(md_lines) + "\n", encoding="utf-8")

print(json.dumps(report, indent=2))
print("Saved:", REPORT_JSON)
print("Saved:", REPORT_MD)
print("Saved:", PER_CLASS_CSV)
print("Saved:", DATASET_SUMMARY_JSON)
print("Saved:", DATASET_OVERVIEW_PNG)
print("Saved:", CLASS_DIST_PNG)
print("Saved:", BBOX_HIST_PNG)
if primary_confusion is not None:
    print("Saved:", CONFUSION_CSV)
